# **K-Nearest Neighbors (KNN)**

## Introducción

K-Nearest Neighbors (KNN) es uno de los algoritmos de machine learning más utilizados en la industria gracias a su **simplicidad y efectividad**.

**Características clave:**
- **Algoritmo no paramétrico**: No asume ninguna distribución específica de los datos
- **Aprendizaje basado en instancias**: Aprende directamente de los datos
- **Aprendizaje supervisado**: Puede usarse tanto para clasificación como para regresión
- **Lazy learning**: Sin fase de training: almacena los datos y los usa directamente para predecir

## Cómo Funciona KNN

Dado un dataset de training, para cada muestra a predecir, KNN:

1. **Calcula la distancia** desde el punto de prueba a todos los puntos de training
2. **Encuentra los K vecinos más cercanos** (las K muestras de training más próximas)
3. **Para clasificación**: Asigna la etiqueta más común entre los K vecinos
4. **Para regresión**: Asigna el valor promedio de los K vecinos

![KNN Example](../img/knn.png)

En la imagen anterior:
- Con **K=3**: El punto verde se clasificaría como "triángulo rojo" (2 triángulos rojos vs 1 cuadrado azul)
- Con **K=5**: El punto verde se clasificaría como "cuadrado azul" (3 cuadrados azules vs 2 triángulos rojos)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification, make_regression

## Características del Lazy Learning

KNN es un algoritmo de **lazy learning** (o **aprendizaje basado en instancias**), lo que significa que **aprende recordando, no resumiendo**. A diferencia de los algoritmos tradicionales de "eager learning" que extraen patrones y fijan valores de parámetros internos durante el training, **KNN almacena todo el dataset de training y lo usa como "base de conocimiento"** para hacer predicciones. Esto puede ser una desventaja, ya que **cada vez que se necesita hacer una predicción, se debe consultar todo el dataset, lo que requiere recursos significativos de memoria y procesamiento**.

Así, la fase de "training" es muy ligera ya que consiste únicamente en almacenar los datos de training. Sin embargo, la predicción es computacionalmente costosa, ya que se debe calcular la distancia de cada punto de prueba a todos los puntos de training. Por ello, **KNN es un algoritmo lento y no es adecuado para datasets grandes**.

## Ejemplo Simple de Clasificación

Empecemos con un ejemplo extremadamente sencillo para entender cómo funciona la clasificación con KNN.

In [ ]:
# Crear un dataset muy simple: 6 puntos en 2D
# Features: coordenadas [x, y]
X_train = np.array([
    [1, 1],   # Clase 0 (Rojo)
    [1.5, 2], # Clase 0 (Rojo)
    [2, 1],   # Clase 0 (Rojo)
    [6, 6],   # Clase 1 (Azul)
    [6.5, 5], # Clase 1 (Azul)
    [7, 6]    # Clase 1 (Azul)
])

# Etiquetas: 0 = Rojo, 1 = Azul
y_train = np.array([0, 0, 0, 1, 1, 1])

# Nuevo punto a clasificar
X_test = np.array([[5, 4]])

print("Datos de training:")
print("Puntos Clase 0 (Rojo):", X_train[y_train == 0])
print("Puntos Clase 1 (Azul):", X_train[y_train == 1])
print("\nPunto de prueba a clasificar:", X_test[0])

In [ ]:
# Visualizar los datos
plt.figure(figsize=(8, 6))

# Graficar puntos de training
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], 
           c='red', s=200, marker='o', edgecolors='k', linewidth=2, label='Clase 0 (Rojo)', alpha=0.7)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], 
           c='blue', s=200, marker='s', edgecolors='k', linewidth=2, label='Clase 1 (Azul)', alpha=0.7)

# Graficar punto de prueba
plt.scatter(X_test[:, 0], X_test[:, 1], 
           c='green', s=300, marker='*', edgecolors='k', linewidth=2, label='Punto de prueba', zorder=5)

# Agregar etiquetas a los puntos
for i, (x, y) in enumerate(X_train):
    plt.annotate(f'P{i+1}', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')

plt.xlabel('Feature 1 (x)', fontsize=12)
plt.ylabel('Feature 2 (y)', fontsize=12)
plt.title('Ejemplo Simple de Clasificación KNN', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

### Calcular Distancias Manualmente

Calculemos manualmente la distancia desde el punto de prueba a cada punto de training usando la **distancia Euclidiana**:

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

In [ ]:
# Calcular distancias desde el punto de prueba a todos los puntos de training
test_point = X_test[0]
distances = []

print(f"Punto de prueba: {test_point}\n")
print(f"{'Punto':<8} {'Coordenadas':<15} {'Clase':<10} {'Cálculo de Distancia':<40} {'Distancia'}")
print("="*95)

for i, (point, label) in enumerate(zip(X_train, y_train)):
    dist = np.sqrt((point[0] - test_point[0])**2 + (point[1] - test_point[1])**2)
    distances.append(dist)
    calc_str = f"√[({point[0]}-{test_point[0]})² + ({point[1]}-{test_point[1]})²]"
    print(f"P{i+1:<7} {str(point):<15} {label:<10} {calc_str:<40} {dist:.3f}")

# Ordenar por distancia
sorted_indices = np.argsort(distances)
print("\n" + "="*95)
print("Puntos ordenados por distancia:")
print("="*95)
for rank, idx in enumerate(sorted_indices, 1):
    print(f"{rank}. P{idx+1} (Clase {y_train[idx]}) - Distancia: {distances[idx]:.3f}")

### Aplicar KNN con Diferentes Valores de K

In [ ]:
# Probar diferentes valores de K
k_values = [1, 3, 5]

print("Predicciones KNN con diferentes valores de K:\n")
print("="*80)

for k in k_values:
    # Crear y entrenar el clasificador KNN
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    # Predecir
    prediction = knn.predict(X_test)[0]
    probabilities = knn.predict_proba(X_test)[0]
    
    # Encontrar los K vecinos más cercanos
    sorted_indices = np.argsort(distances)[:k]
    neighbor_classes = y_train[sorted_indices]
    
    print(f"\nK = {k}:")
    print(f"  Vecinos más cercanos: ", end="")
    for idx in sorted_indices:
        print(f"P{idx+1}(Clase {y_train[idx]}, dist={distances[idx]:.3f})  ", end="")
    print(f"\n  Clases de vecinos: {neighbor_classes}")
    print(f"  Conteo de votos: Clase 0: {np.sum(neighbor_classes == 0)}, Clase 1: {np.sum(neighbor_classes == 1)}")
    print(f"  Clase predicha: {prediction} ({'Rojo' if prediction == 0 else 'Azul'})")
    print(f"  Probabilidades: Clase 0: {probabilities[0]:.2%}, Clase 1: {probabilities[1]:.2%}")

print("\n" + "="*80)

### Visualizar los K Vecinos Más Cercanos

In [ ]:
# Visualizar con diferentes valores de K
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, k in enumerate([1, 3, 5]):
    ax = axes[idx]
    
    # Graficar puntos de training
    ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], 
              c='red', s=200, marker='o', edgecolors='k', linewidth=2, label='Clase 0', alpha=0.7)
    ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], 
              c='blue', s=200, marker='s', edgecolors='k', linewidth=2, label='Clase 1', alpha=0.7)
    
    # Graficar punto de prueba
    ax.scatter(X_test[:, 0], X_test[:, 1], 
              c='green', s=300, marker='*', edgecolors='k', linewidth=2, label='Punto de prueba', zorder=5)
    
    # Trazar líneas hacia los K vecinos más cercanos
    sorted_indices = np.argsort(distances)[:k]
    for neighbor_idx in sorted_indices:
        ax.plot([X_test[0, 0], X_train[neighbor_idx, 0]], 
               [X_test[0, 1], X_train[neighbor_idx, 1]], 
               'g--', linewidth=2, alpha=0.6)
    
    # Hacer predicción
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    prediction = knn.predict(X_test)[0]
    pred_color = 'Rojo' if prediction == 0 else 'Azul'
    
    ax.set_xlabel('Feature 1 (x)', fontsize=11)
    ax.set_ylabel('Feature 2 (y)', fontsize=11)
    ax.set_title(f'K = {k}\nPredicción: {pred_color}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Clasificación con un Dataset más Grande

Veamos cómo funciona KNN con un dataset más realista y visualicemos las fronteras de decisión.

In [ ]:
# Generar un dataset sintético
np.random.seed(42)
X, y = make_classification(n_samples=100, n_features=2, n_redundant=0, n_informative=2,
                          n_clusters_per_class=1, flip_y=0.1, random_state=42)

# Dividir en training y prueba
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño del conjunto de training: {len(X_train_large)}")
print(f"Tamaño del conjunto de prueba: {len(X_test_large)}")
print(f"\nDistribución de clases en training:")
print(f"  Clase 0: {np.sum(y_train_large == 0)} muestras")
print(f"  Clase 1: {np.sum(y_train_large == 1)} muestras")

In [ ]:
# Entrenar modelos con diferentes valores de K y comparar
from sklearn.metrics import accuracy_score

k_values = [1, 3, 5, 10, 20]
accuracies = []

print("Rendimiento del Modelo con Diferentes Valores de K:\n")
print(f"{'K':<5} {'Precisión en Training':<20} {'Precisión en Prueba'}")
print("="*50)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_large, y_train_large)
    
    train_acc = accuracy_score(y_train_large, knn.predict(X_train_large))
    test_acc = accuracy_score(y_test_large, knn.predict(X_test_large))
    accuracies.append(test_acc)
    
    print(f"{k:<5} {train_acc:<20.2%} {test_acc:.2%}")

print("="*50)
best_k = k_values[np.argmax(accuracies)]
print(f"\nMejor valor de K: {best_k} (Precisión en prueba: {max(accuracies):.2%})")

In [ ]:
# Visualizar fronteras de decisión para diferentes valores de K
from matplotlib.colors import ListedColormap

def plot_decision_boundary(X, y, k, ax):
    """Grafica la frontera de decisión del clasificador KNN"""
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    
    # Crear malla
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predecir sobre la malla
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Graficar
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#0000FF'])
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolor='k', s=50, alpha=0.8)
    ax.set_title(f'K = {k}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Graficar fronteras de decisión
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
k_values_viz = [1, 3, 5, 10, 20, 50]

for idx, k in enumerate(k_values_viz):
    ax = axes[idx // 3, idx % 3]
    plot_decision_boundary(X_train_large, y_train_large, k, ax)

plt.suptitle('Fronteras de Decisión KNN con Diferentes Valores de K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObservaciones:")
print("• K=1: Frontera muy compleja, propensa a overfitting (se ajusta demasiado a los datos de training)")
print("• K=5-10: Fronteras más suaves, mejor generalización")
print("• K=50: Demasiado suave, puede causar underfitting (pierde patrones importantes)")

## La Importancia Crítica del Escalado de Features en KNN

Una de las consideraciones más importantes al usar KNN es el **escalado de features**. Dado que KNN usa métricas de distancia (como la distancia Euclidiana) para encontrar los vecinos más cercanos, los features con escalas más grandes dominarán el cálculo de distancia, **opacando** efectivamente a los features con escalas más pequeñas.

### Por qué el Escalado es Importante

Considera dos features:
- **Feature 1**: Edad (rango: 0-100)
- **Feature 2**: Ingreso (rango: 0-100,000)

Al calcular la distancia Euclidiana, una diferencia de $10,000 en ingreso dominará completamente una diferencia de 10 años en edad, aunque ambas diferencias sean igualmente importantes para la clasificación.

Demostraremos esto con un ejemplo visual.

In [ ]:
# Generar datos sintéticos con dos features de escalas muy diferentes
np.random.seed(0)

# Clase 0: centrada en (5, 50)
X_class0_f1 = np.random.normal(5, 1.5, 15)    # Feature 1: escala pequeña (0-10)
X_class0_f2 = np.random.normal(50, 15, 15)    # Feature 2: escala grande (0-100)

# Clase 1: centrada en (8, 70)
X_class1_f1 = np.random.normal(8, 1.5, 15)    # Feature 1: escala pequeña (0-10)
X_class1_f2 = np.random.normal(70, 15, 15)    # Feature 2: escala grande (0-100)

# Combinar los datos
X_unscaled = np.column_stack([
    np.concatenate([X_class0_f1, X_class1_f1]),
    np.concatenate([X_class0_f2, X_class1_f2])
])
y = np.concatenate([np.zeros(15), np.ones(15)])

print(f"Rango del Feature 1: [{X_unscaled[:, 0].min():.2f}, {X_unscaled[:, 0].max():.2f}]")
print(f"Rango del Feature 2: [{X_unscaled[:, 1].min():.2f}, {X_unscaled[:, 1].max():.2f}]")
print(f"\nRatio de escala: {(X_unscaled[:, 1].max() - X_unscaled[:, 1].min()) / (X_unscaled[:, 0].max() - X_unscaled[:, 0].min()):.1f}x")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Crear un punto de prueba
test_point = np.array([[6.5, 60]])

# Entrenar KNN con datos sin escalar
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_unscaled, y)

# Encontrar vecinos para el punto de prueba
distances_unscaled, indices_unscaled = knn_unscaled.kneighbors(test_point)

# Escalar los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled)
test_point_scaled = scaler.transform(test_point)

# Entrenar KNN con datos escalados
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_scaled, y)

# Encontrar vecinos para el punto de prueba escalado
distances_scaled, indices_scaled = knn_scaled.kneighbors(test_point_scaled)

# Visualizar la diferencia
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Datos sin escalar
ax1 = axes[0]
ax1.scatter(X_unscaled[y==0, 0], X_unscaled[y==0, 1], 
           c='red', s=100, alpha=0.6, label='Clase 0', edgecolors='k')
ax1.scatter(X_unscaled[y==1, 0], X_unscaled[y==1, 1], 
           c='blue', s=100, alpha=0.6, label='Clase 1', edgecolors='k')
ax1.scatter(test_point[0, 0], test_point[0, 1], 
           c='green', s=300, marker='*', label='Punto de Prueba', 
           edgecolors='k', linewidths=2, zorder=5)

# Resaltar vecinos en datos sin escalar
neighbors_unscaled = X_unscaled[indices_unscaled[0]]
ax1.scatter(neighbors_unscaled[:, 0], neighbors_unscaled[:, 1], 
           s=400, facecolors='none', edgecolors='orange', 
           linewidths=3, label='5 Vecinos Más Cercanos')

# Trazar líneas hacia los vecinos
for neighbor in neighbors_unscaled:
    ax1.plot([test_point[0, 0], neighbor[0]], 
            [test_point[0, 1], neighbor[1]], 
            'orange', linestyle='--', alpha=0.5, linewidth=1.5)

ax1.set_xlabel('Feature 1 (Escala Pequeña: 0-10)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Feature 2 (Escala Grande: 0-100)', fontsize=12, fontweight='bold')
ax1.set_title('SIN Escalado\n(Feature 2 domina el cálculo de distancia)', 
             fontsize=13, fontweight='bold', color='red')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Agregar anotación mostrando el problema
prediction_unscaled = knn_unscaled.predict(test_point)[0]
ax1.text(0.02, 0.98, f'Predicción: Clase {int(prediction_unscaled)}', 
        transform=ax1.transAxes, fontsize=11, fontweight='bold',
        verticalalignment='top', bbox=dict(boxstyle='round', 
        facecolor='wheat', alpha=0.8))

# Gráfica 2: Datos escalados
ax2 = axes[1]
ax2.scatter(X_scaled[y==0, 0], X_scaled[y==0, 1], 
           c='red', s=100, alpha=0.6, label='Clase 0', edgecolors='k')
ax2.scatter(X_scaled[y==1, 0], X_scaled[y==1, 1], 
           c='blue', s=100, alpha=0.6, label='Clase 1', edgecolors='k')
ax2.scatter(test_point_scaled[0, 0], test_point_scaled[0, 1], 
           c='green', s=300, marker='*', label='Punto de Prueba', 
           edgecolors='k', linewidths=2, zorder=5)

# Resaltar vecinos en datos escalados
neighbors_scaled = X_scaled[indices_scaled[0]]
ax2.scatter(neighbors_scaled[:, 0], neighbors_scaled[:, 1], 
           s=400, facecolors='none', edgecolors='lime', 
           linewidths=3, label='5 Vecinos Más Cercanos')

# Trazar líneas hacia los vecinos
for neighbor in neighbors_scaled:
    ax2.plot([test_point_scaled[0, 0], neighbor[0]], 
            [test_point_scaled[0, 1], neighbor[1]], 
            'lime', linestyle='--', alpha=0.5, linewidth=1.5)

ax2.set_xlabel('Feature 1 (Estandarizado)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Feature 2 (Estandarizado)', fontsize=12, fontweight='bold')
ax2.set_title('CON Escalado (StandardScaler)\n(Ambos features contribuyen por igual)', 
             fontsize=13, fontweight='bold', color='green')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Agregar anotación
prediction_scaled = knn_scaled.predict(test_point_scaled)[0]
ax2.text(0.02, 0.98, f'Predicción: Clase {int(prediction_scaled)}', 
        transform=ax2.transAxes, fontsize=11, fontweight='bold',
        verticalalignment='top', bbox=dict(boxstyle='round', 
        facecolor='lightgreen', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\nPredicción SIN escalado: Clase {int(prediction_unscaled)}")
print(f"Predicción CON escalado: Clase {int(prediction_scaled)}")
print(f"\n¡Las predicciones {'DIFIEREN' if prediction_unscaled != prediction_scaled else 'son IGUALES'}!")

### Análisis de los Resultados

Como puedes ver en las visualizaciones anteriores:

**Sin Escalado (Gráfica Izquierda)**:
- El Feature 2 tiene un rango mucho más grande (0-100) que el Feature 1 (0-10)
- El cálculo de distancia es dominado por el Feature 2
- Los vecinos más cercanos se seleccionan principalmente por cercanía en la dirección vertical (Feature 2)
- El Feature 1 queda **opacado** y contribuye muy poco a la decisión

**Con Escalado (Gráfica Derecha)**:
- Ambos features están estandarizados para tener rangos similares
- El cálculo de distancia considera ambos features por igual
- Los vecinos más cercanos se seleccionan basándose en la proximidad real en ambas dimensiones
- Ambos features contribuyen significativamente a la decisión de clasificación

### Conclusiones Clave

1. **Siempre escala tus features** cuando uses KNN (o cualquier algoritmo basado en distancias)
2. Métodos de escalado comunes:
   - **StandardScaler**: Estandariza los features para tener media=0 y varianza=1
   - **MinMaxScaler**: Escala los features a un rango fijo (generalmente 0-1)
   - **RobustScaler**: Usa la mediana y el IQR, menos sensible a valores atípicos
3. **Predicciones diferentes** pueden resultar de datos escalados vs. sin escalar
4. El escalado asegura que **todos los features contribuyan equitativamente** al cálculo de distancia

In [ ]:
# Veamos las distancias reales para entender el efecto de opacamiento
print("Contribuciones de Distancia para el Primer Vecino:\n" + "="*60)

# Distancias sin escalar
first_neighbor_unscaled = X_unscaled[indices_unscaled[0][0]]
dist_f1_unscaled = abs(test_point[0, 0] - first_neighbor_unscaled[0])
dist_f2_unscaled = abs(test_point[0, 1] - first_neighbor_unscaled[1])
total_dist_unscaled = np.sqrt(dist_f1_unscaled**2 + dist_f2_unscaled**2)

print("\nSIN escalado:")
print(f"  Contribución del Feature 1: {dist_f1_unscaled:.3f}")
print(f"  Contribución del Feature 2: {dist_f2_unscaled:.3f}")
print(f"  Distancia total: {total_dist_unscaled:.3f}")
print(f"  Dominancia del Feature 2: {(dist_f2_unscaled/total_dist_unscaled)*100:.1f}% de la distancia total")

# Distancias escaladas
first_neighbor_scaled = X_scaled[indices_scaled[0][0]]
dist_f1_scaled = abs(test_point_scaled[0, 0] - first_neighbor_scaled[0])
dist_f2_scaled = abs(test_point_scaled[0, 1] - first_neighbor_scaled[1])
total_dist_scaled = np.sqrt(dist_f1_scaled**2 + dist_f2_scaled**2)

print("\nCON escalado:")
print(f"  Contribución del Feature 1: {dist_f1_scaled:.3f}")
print(f"  Contribución del Feature 2: {dist_f2_scaled:.3f}")
print(f"  Distancia total: {total_dist_scaled:.3f}")
print(f"  Features balanceados: ~{(max(dist_f1_scaled, dist_f2_scaled)/total_dist_scaled)*100:.1f}% vs ~{(min(dist_f1_scaled, dist_f2_scaled)/total_dist_scaled)*100:.1f}%")

## Ejemplo Simple de Regresión

En regresión, KNN asigna el **valor promedio** de los K vecinos más cercanos al punto de prueba.

In [ ]:
# Crear un ejemplo de regresión simple en 1D
X_reg_train = np.array([[1], [2], [3], [5], [6], [7]])
y_reg_train = np.array([2, 3, 3.5, 5, 5.5, 6])

# Punto a predecir
X_reg_test = np.array([[4]])

print("Datos de training:")
for x, y in zip(X_reg_train.ravel(), y_reg_train):
    print(f"  x = {x}, y = {y}")
print(f"\nPredecir y para x = {X_reg_test[0, 0]}")

In [ ]:
# Calcular distancias y hacer predicciones manualmente
test_x = X_reg_test[0, 0]
distances_reg = np.abs(X_reg_train.ravel() - test_x)

print(f"\nDistancias desde x = {test_x}:\n")
print(f"{'Punto de Training':<20} {'Distancia':<15} {'Valor y'}")
print("="*50)
for i, (x, dist, y) in enumerate(zip(X_reg_train.ravel(), distances_reg, y_reg_train)):
    print(f"x = {x:<17} {dist:<15.1f} {y}")

# Ordenar por distancia
sorted_indices_reg = np.argsort(distances_reg)

print("\n" + "="*50)
print("Predicciones con diferentes valores de K:\n")

for k in [1, 2, 3]:
    # Obtener K vecinos más cercanos
    nearest_indices = sorted_indices_reg[:k]
    nearest_x = X_reg_train[nearest_indices].ravel()
    nearest_y = y_reg_train[nearest_indices]
    
    # Calcular predicción (promedio de los K vecinos más cercanos)
    prediction_manual = np.mean(nearest_y)
    
    # Usando sklearn
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_reg_train, y_reg_train)
    prediction_sklearn = knn_reg.predict(X_reg_test)[0]
    
    print(f"K = {k}:")
    print(f"  Vecinos más cercanos: x = {list(nearest_x)}, y = {list(nearest_y)}")
    print(f"  Predicción = promedio({list(nearest_y)}) = {prediction_manual:.2f}")
    print(f"  Predicción sklearn: {prediction_sklearn:.2f}")
    print()

In [ ]:
# Visualizar regresión con diferentes valores de K
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Crear un rango para predicciones suaves
X_range = np.linspace(0.5, 7.5, 100).reshape(-1, 1)

for idx, k in enumerate([1, 2, 3]):
    ax = axes[idx]
    
    # Entrenar modelo
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_reg_train, y_reg_train)
    
    # Predecir sobre el rango
    y_pred_range = knn_reg.predict(X_range)
    
    # Graficar datos de training
    ax.scatter(X_reg_train, y_reg_train, c='blue', s=150, edgecolors='k', 
              linewidth=2, label='Datos de training', zorder=3)
    
    # Graficar línea de predicción
    ax.plot(X_range, y_pred_range, 'r-', linewidth=2, label='Predicción KNN', alpha=0.8)
    
    # Resaltar punto de prueba
    test_pred = knn_reg.predict(X_reg_test)[0]
    ax.scatter(X_reg_test, test_pred, c='green', s=300, marker='*', 
              edgecolors='k', linewidth=2, label=f'Predicción (y={test_pred:.2f})', zorder=4)
    
    # Trazar líneas hacia los vecinos más cercanos
    sorted_indices_reg = np.argsort(distances_reg)[:k]
    for neighbor_idx in sorted_indices_reg:
        ax.plot([X_reg_test[0, 0], X_reg_train[neighbor_idx, 0]], 
               [test_pred, y_reg_train[neighbor_idx]], 
               'g--', linewidth=1.5, alpha=0.5)
    
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_title(f'K = {k}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Regresión KNN con Diferentes Valores de K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Regresión con un Dataset Real

Apliquemos la regresión KNN a un escenario más realista.

In [ ]:
# Generar datos de regresión sintéticos
np.random.seed(42)
X_reg_large = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg_large = np.sin(X_reg_large).ravel() + np.random.normal(0, 0.1, X_reg_large.shape[0])

# Dividir datos
X_reg_train_large = X_reg_large[:60]
y_reg_train_large = y_reg_large[:60]
X_reg_test_large = X_reg_large[60:]
y_reg_test_large = y_reg_large[60:]

print(f"Tamaño del conjunto de training: {len(X_reg_train_large)}")
print(f"Tamaño del conjunto de prueba: {len(X_reg_test_large)}")

In [ ]:
# Comparar diferentes valores de K
from sklearn.metrics import mean_squared_error, r2_score

k_values_reg = [1, 3, 5, 10, 20]

print("Rendimiento de Regresión con Diferentes Valores de K:\n")
print(f"{'K':<5} {'MSE Training':<15} {'MSE Prueba':<15} {'R² Prueba'}")
print("="*55)

for k in k_values_reg:
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_reg_train_large, y_reg_train_large)
    
    train_pred = knn_reg.predict(X_reg_train_large)
    test_pred = knn_reg.predict(X_reg_test_large)
    
    train_mse = mean_squared_error(y_reg_train_large, train_pred)
    test_mse = mean_squared_error(y_reg_test_large, test_pred)
    test_r2 = r2_score(y_reg_test_large, test_pred)
    
    print(f"{k:<5} {train_mse:<15.4f} {test_mse:<15.4f} {test_r2:.4f}")

print("="*55)

In [ ]:
# Visualizar ajustes de regresión
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
k_values_viz_reg = [1, 3, 5, 10, 20, 30]

X_plot = np.linspace(0, 5, 500).reshape(-1, 1)

for idx, k in enumerate(k_values_viz_reg):
    ax = axes[idx // 3, idx % 3]
    
    # Entrenar modelo
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_reg_train_large, y_reg_train_large)
    
    # Predecir
    y_plot = knn_reg.predict(X_plot)
    test_pred = knn_reg.predict(X_reg_test_large)
    test_mse = mean_squared_error(y_reg_test_large, test_pred)
    
    # Graficar
    ax.scatter(X_reg_train_large, y_reg_train_large, c='blue', s=50, 
              edgecolors='k', alpha=0.6, label='Datos de training')
    ax.scatter(X_reg_test_large, y_reg_test_large, c='green', s=50, 
              edgecolors='k', alpha=0.6, label='Datos de prueba')
    ax.plot(X_plot, y_plot, 'r-', linewidth=2, label='Predicción KNN')
    
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(f'K = {k}\nMSE Prueba = {test_mse:.4f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Regresión KNN: Efecto de K en la Suavidad del Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObservaciones:")
print("• K=1: Muy irregular, sigue demasiado los datos de training (overfitting)")
print("• K=5-10: Curva suave que captura bien el patrón")
print("• K=30: Demasiado suave, pierde el patrón sinusoidal (underfitting)")

## Métricas de Distancia

KNN usa la distancia para encontrar vecinos. Las métricas de distancia más comunes son:

1. **Distancia Euclidiana** (por defecto): $d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$
2. **Distancia Manhattan**: $d = \sum_{i=1}^{n}|x_i - y_i|$
3. **Distancia Minkowski**: $d = (\sum_{i=1}^{n}|x_i - y_i|^p)^{1/p}$ (generaliza la Euclidiana y la Manhattan)

In [ ]:
# Comparar diferentes métricas de distancia
from sklearn.metrics import accuracy_score

metrics = ['euclidean', 'manhattan', 'minkowski']
k = 5

print(f"Comparación de Métricas de Distancia (K = {k}):\n")
print(f"{'Métrica':<15} {'Precisión Training':<20} {'Precisión Prueba'}")
print("="*55)

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
    knn.fit(X_train_large, y_train_large)
    
    train_acc = accuracy_score(y_train_large, knn.predict(X_train_large))
    test_acc = accuracy_score(y_test_large, knn.predict(X_test_large))
    
    print(f"{metric:<15} {train_acc:<20.2%} {test_acc:.2%}")

print("="*55)
print("\nNota: Diferentes métricas pueden funcionar mejor en distintos datasets.")
print("Euclidiana es la más común para features continuos.")

## Características Clave de KNN

### Ventajas:
- ✅ **Simple de entender e implementar**
- ✅ **Sin fase de training** (almacena los datos directamente)
- ✅ **Funciona bien con datasets pequeños**
- ✅ **Sin suposiciones sobre la distribución de datos** (no paramétrico)
- ✅ **Puede usarse tanto para clasificación como para regresión**

### Desventajas:
- ❌ **Predicción lenta** (debe calcular la distancia a todos los puntos de training)
- ❌ **Alto consumo de memoria** (almacena todo el dataset)
- ❌ **No adecuado para datasets grandes** (maldición de la dimensionalidad)
- ❌ **Sensible al escalado de features** (los features con rangos grandes dominan el cálculo de distancia)
- ❌ **Sensible a features irrelevantes**
- ❌ **La elección de K es crítica**

### Complejidad Computacional:
- **Training:** O(1) - solo almacena los datos
- **Predicción:** O(n × d) - donde n = número de muestras de training, d = número de features

¡Esto es lo opuesto a la mayoría de los algoritmos de ML donde el training es costoso pero la predicción es rápida!

![KNN Decision Surface Animation](https://upload.wikimedia.org/wikipedia/commons/7/78/KNN_decision_surface_animation.gif)

## Elegir el Valor Correcto de K

**Guías para seleccionar K:**

1. **K = 1**: Muy flexible, alta varianza, propenso a overfitting
2. **K = √n** (donde n es el número de muestras de training): Regla general común
3. **K = grande**: Muy suave, alto sesgo, propenso a underfitting
4. **Usa validación cruzada** para encontrar el K óptimo para tu dataset
5. **K impar para clasificación binaria** para evitar empates

**Compromiso:**
- K pequeño → modelo más complejo → overfitting
- K grande → modelo más simple → underfitting

In [ ]:
# Encontrar el K óptimo usando validación cruzada
from sklearn.model_selection import cross_val_score

k_range = range(1, 31)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_large, y_train_large, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

# Graficar resultados
plt.figure(figsize=(12, 6))
plt.plot(k_range, k_scores, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Valor de K', fontsize=12)
plt.ylabel('Precisión con Validación Cruzada', fontsize=12)
plt.title('Encontrar el K Óptimo con Validación Cruzada', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Marcar el mejor K
best_k_cv = k_range[np.argmax(k_scores)]
plt.axvline(x=best_k_cv, color='r', linestyle='--', linewidth=2, label=f'K Óptimo = {best_k_cv}')
plt.legend(fontsize=11)
plt.show()

print(f"Valor óptimo de K: {best_k_cv}")
print(f"Precisión con validación cruzada: {max(k_scores):.2%}")

## Resumen

KNN es un **algoritmo simple pero poderoso** que:
- Hace predicciones basándose en los K ejemplos de training más cercanos
- Usa **votación mayoritaria** para clasificación
- Usa el **promedio** para regresión
- No tiene fase de training (lazy learning)
- Es sensible a la elección de K y a la métrica de distancia
- Funciona mejor con datasets de tamaño pequeño a mediano
- Requiere escalado de features para mejores resultados

**Cuándo usar KNN:**
- Datasets pequeños a medianos
- Cuando necesitas un modelo de referencia simple
- Cuando la frontera de decisión es muy irregular
- Cuando tienes datos de baja dimensionalidad (pocos features)

**Cuándo NO usar KNN:**
- Datasets grandes (predicción lenta)
- Datos de alta dimensionalidad (maldición de la dimensionalidad)
- Requisitos de predicción en tiempo real
- Cuando la memoria es limitada